# Ricerca Non Informata

In [28]:
# questo è qualcosa che eseguo ogni volta all'inizio del notebook
# per poter accedere ai moduli presenti in scripts
from pathlib import Path
import sys

# 1. Risalgo dalla cartella corrente finché non trovo quella che contiene "scripts"
ROOT = Path.cwd()
while not (ROOT / "scripts").exists():
    ROOT = ROOT.parent

# 2. Aggiungo la root al path di Python
sys.path.insert(0, str(ROOT))

# 3. Importo i moduli del corso come scripts.nome_modulo
from scripts.environment import *
from scripts.agent import Agent
from scripts.runner import run_episode

In questo notebook lavoreremo con la ricerca non informata.\
Faremo un semplice esercizio di riscaldamento in cui cercheremo di capire l'importanza di usare la struttura dati giusta a seconda del caso specifico, e poi proveremo a implementare un agente di pianificazione nel nostro Mondo dell'Aspirapolvere.

Per iniziare, facciamo un breve ripasso della teoria dietro la ricerca classica.

## L'agente di pianificazione e il problema di ricerca

A differenza di un agente riflesso che sceglie le azioni basandosi unicamente sul percetto corrente, un agente di pianificazione usa un modello interno per esplorare le potenziali sequenze di azioni.

Assumiamo, da qui in avanti, che l'ambiente sia a singolo agente, completamente osservabile, statico (nessun cambiamento esterno durante la pianificazione), con stati discreti e azioni deterministiche, e che segua una struttura di task sequenziale in cui un'azione influenza le possibilità future.

Un problema di ricerca è definito dalle seguenti componenti:

* Spazio degli Stati ($S$): l'insieme di tutti gli stati possibili.
* Azioni Disponibili ($A$): le azioni disponibili per un dato stato $s$, denotate come $A: S \rightarrow 2^{\mathcal{A}}$.
* Modello di Transizione ($T$): restituisce lo stato successore dato uno stato e un'azione, $T: S \times \mathcal{A} \rightarrow S$.
* Costo del Passo ($C$): il costo numerico di un'azione, $c: S \times \mathcal{A} \times S \rightarrow \mathbb{R}_{\ge 0}$.
* Stato Iniziale ($s_0$): lo stato di partenza dell'agente.
* Test dell'Obiettivo: una funzione che determina se un dato stato è uno stato obiettivo ($S \rightarrow \{0, 1\}$).

Definiamo quindi:

* **Piano**: è una sequenza di azioni $\pi = (a_0, \dots, a_{k-1})$ che porta dallo stato iniziale a uno stato obiettivo.
* **Costo del percorso**: è la somma dei costi dei passi lungo il percorso: $g(\pi) = \sum_{i=0}^{k-1} c(s_i, a_i, s_{i+1})$. Un piano ottimo minimizza questo costo.

L'obiettivo di solito non è solo trovare un **piano valido**, ma quello con il **costo del percorso minimo**.

La pianificazione si basa su un modello dell'ambiente che mantiene informazioni sufficienti. In particolare, uno stato deve contenere abbastanza informazioni per determinare le azioni legali, gli stati successori, i costi dei passi e se l'obiettivo è raggiunto. I costi passati appartengono al nodo di ricerca, non allo stato in sé, a meno che non influenzino le possibilità future.

Il numero di stati possibili dipende dal tipo di problemi e da come i diversi stati interagiscono tra loro.

Il modello stesso può essere rappresentato di solito con due tipi di rappresentazione:

* **Grafo dello Spazio degli Stati**: contiene un vertice per ogni stato e archi che rappresentano azioni legali. Loop e cicli possono creare percorsi infiniti.
* **Albero di Ricerca**: srotola il grafo dello spazio degli stati in un albero in cui ogni nodo rappresenta un intero percorso dallo stato iniziale. Nodi diversi possono puntare allo stesso stato sottostante.

La **frontiera** è un importante oggetto ausiliario usato durante l'esplorazione degli stati. Il suo compito è memorizzare i nodi candidati in attesa di essere selezionati. La sua struttura dati sottostante definisce la strategia di ricerca (quello che chiamiamo "l'algoritmo"):

* Stack (LIFO): usato per la Depth-First Search (DFS).
* Queue (FIFO): usata per la Breadth-First Search (BFS).
* Coda a priorità minima: usata per la Uniform-Cost Search (UCS).

## Algoritmi

Giusto per ripassare, riportiamo gli algoritmi visti a lezione per la ricerca non informata:

* **Depth First Search** (DFS)
* **Breadth First Search** (BFS)
* **Iterative Deepening Search** (IDS)
* **Uniform Cost Search** (UCS)

Li abbiamo anche valutati rispetto a completezza, ottimalità, complessità temporale e complessità spaziale.

| Criterio / Metodo | DFS | BFS | IDS | UCS |
| :--- | :--- | :--- | :--- | :--- |
| **Completo?** | No (in generale) | Sì (branching finito) | Sì (branching finito) | Sì (sotto le condizioni enunciate) |
| **Ottimo rispetto al costo?** | No | Sì (costi positivi uguali) | Sì (costi positivi uguali) | Sì (costi non negativi) |
| **Complessità temporale** | O(b^m) | O(b^(d+1)) | O(b^d) | O(b^(1 + floor(C*/epsilon))) |
| **Complessità spaziale** | O(bm) | O(b^(d+1)) | O(bd) | O(b^(1 + floor(C*/epsilon))) |

## Esercizio di riscaldamento

Vogliamo capire come scegliere uno Stack (LIFO), una Queue (FIFO) o una Coda a Priorità Minima cambia l'ordine in cui un algoritmo di ricerca espande i nodi.

Completa il seguente frammento di codice inserendo gli elementi salvati in `items` dentro `stack`, `queue` e `pq`. Usa:

* una **lista standard di Python** come Stack (LIFO)
* **collections.deque** come Queue (FIFO)
* **heapq** come Coda a Priorità Minima (UCS)

In [29]:
from collections import deque
import heapq

items = [(4, "A"), (1, "B"), (3, "C")]

# Stack (LIFO) -> DFS

# 1. Uso una lista Python come stack
stack = []
for item in items:  
    # 2. Aggiungo in cima con append
    stack.append(item)
# 3. Estraggo dal fondo (LIFO) con pop
print("DFS Next Out:", stack.pop())


# Queue (FIFO) -> BFS

# 4. Uso una deque come coda
queue = deque()
for item in items:
    # 5. Aggiungo in fondo con append
    queue.append(item)
# 6. Estraggo dalla testa (FIFO) con popleft
print("BFS Next Out:", queue.popleft())

# Min-Priority Queue -> UCS

# 7. Uso una lista come heap tramite heapq
pq = []
for item in items:
    # 8. heapq.heappush ordina automaticamente in base al primo elemento (costo)
    heapq.heappush(pq, item)
# 9. Estraggo l'elemento con costo minore
print("UCS Next Out:", heapq.heappop(pq))

DFS Next Out: (3, 'C')
BFS Next Out: (4, 'A')
UCS Next Out: (1, 'B')


Un aspetto interessante che vogliamo considerare è anche il fatto che la complessità di ciascuna operazione non è sempre la stessa.

* Stack (LIFO / DFS): inserimenti e rimozioni avvengono in tempo $O(1)$.
* Queue (FIFO / BFS): inserimenti e rimozioni avvengono in tempo $O(1)$ (usando puntatori a testa/coda).
* Coda a Priorità Minima (UCS / Heap): inserimenti e rimozioni avvengono in tempo $O(\log F)$, dove $F$ è il numero di elementi nella frontiera, a causa del riordino del binary heap.

Proviamo a misurarlo!

In [31]:
import random
import time

# Numero di elementi da inserire e rimuovere
n_operations = 100000

# Dati casuali
data = [(random.randint(1, 1000), f"node_{i}") for i in range(n_operations)]

from collections import deque
import heapq

# Stack (LIFO) -> DFS
start_time = time.perf_counter()

# 1. Uso una lista Python come stack
stack = []
for item in data:
    # 2. Inserisco in cima
    stack.append(item)
while stack:
    # 3. Rimuovo la testa (in una lista = ultimo elemento = pop)
    stack.pop()
stack_time = time.perf_counter() - start_time

# Queue (FIFO) -> BFS
start_time = time.perf_counter()

# 4. Uso una deque come coda
queue = deque()
for item in data:
    # 5. Inserisco in fondo
    queue.append(item)
while queue:
    # 6. Rimuovo la testa (in una deque = primo elemento = popleft)
    queue.popleft()
queue_time = time.perf_counter() - start_time

# Min-Priority Queue -> UCS
start_time = time.perf_counter()

# 7. Uso un heap (lista gestita con heapq)
pq = []
for item in data:
    # 8. heapq.heappush ordina automaticamente in base al primo elemento (costo)
    heapq.heappush(pq, item)
while pq:
    # 9. Rimuovo l'elemento con costo minore
    heapq.heappop(pq)
pq_time = time.perf_counter() - start_time

print(f"Risultati del benchmark su {n_operations:,} operazioni")
print(f"Stack (LIFO - Lista): {stack_time:.4f} secondi")
print(f"Queue (FIFO - Deque): {queue_time:.4f} secondi")
print(f"Coda a Priorità (Heap): {pq_time:.4f} secondi")

Risultati del benchmark su 100,000 operazioni
Stack (LIFO - Lista): 0.0157 secondi
Queue (FIFO - Deque): 0.0170 secondi
Coda a Priorità (Heap): 0.3940 secondi


## Agente di Pianificazione

È ora di scrivere il nostro agente di pianificazione!\
Useremo un albero come struttura dati; per farlo, useremo un semplice oggetto ausiliario che chiameremo `SearchNode`.

In [32]:
from collections import deque
import heapq

class SearchNode:
  """Rappresenta un nodo nell'albero di ricerca (un percorso specifico)."""

  def __init__(self, state, parent=None, action=None, path_cost=0.0):
    self.state = state    # Consiglio: nella forma ((x, y), frozenset(dirts))
    self.parent = parent  # SearchNode precedente
    self.action = action  # Azione eseguita per arrivare qui
    self.path_cost = path_cost  # Costo cumulativo (g)

  # Necessario per i confronti della coda a priorità in UCS
  # Questo è il dunder method del "minore di"
  def __lt__(self, other):
    return self.path_cost < other.path_cost

Dovete provare a implementare tutto ciò che abbiamo descritto finora.\
Creerete un metodo `search` che, dato lo stato iniziale e l'ambiente, costruisce l'albero di ricerca restituendo un percorso valido verso lo stato obiettivo.\
Attenzione: dovrete simulare l'ambiente all'interno del metodo di ricerca. Usate gli helper `_get_legal_actions`, `_transition` e `_reconstruct_path`.\
Nota: questa implementazione dell'agente non usa direttamente `env` come helper. Questo può rendere l'agente meno riutilizzabile, ma è un buon modo per separare la logica della pianificazione dall'ambiente stesso!

In [33]:
class PlanningAgent(Agent):

  def __init__(self, env, search_strategy='bfs'):
    super().__init__(env)
    self.env = env
    self.search_strategy = search_strategy
    self.plan = []

  def act(self, observation):
    # 1. Se ho già un piano calcolato, estraggo e restituisco la prossima azione
    if self.plan:
      return self.plan.pop(0)

    # 2. Altrimenti eseguo la ricerca per trovare un piano
    self.plan = self.search()
    if self.plan:
      return self.plan.pop(0)
    return 4  # Fallback a IDLE se non trovo un piano

  def search(self):
    """Routine di ricerca principale: costruisce l'albero di ricerca e trova un percorso verso l'obiettivo."""
    # 1. Costruisco lo stato iniziale dall'ambiente
    initial_pos = self.env.vacuum
    initial_dirts = frozenset(self.env.dirts)
    initial_state = (initial_pos, initial_dirts)

    # 2. Creo il nodo radice
    root_node = SearchNode(state=initial_state, path_cost=0.0)

    # 3. Inizializzo la frontiera in base alla strategia
    #    - BFS: deque (FIFO)
    #    - DFS: list (LIFO stack)
    #    - UCS: heapq (Coda a Priorità Minima)
    if self.search_strategy == 'bfs':
      frontier = deque([root_node])
    elif self.search_strategy == 'dfs':
      frontier = [root_node]
    elif self.search_strategy == 'ucs':
      frontier = []
      heapq.heappush(frontier, root_node)
    else:
      raise ValueError(f"Strategia di ricerca sconosciuta: {self.search_strategy}")
  
    # 4. Inizializzo l'insieme degli stati esplorati
    explored = set()

    while frontier:
      # 5. Estraggo il nodo dalla frontiera in base alla strategia
      if self.search_strategy == 'bfs':
        node = frontier.popleft()
      elif self.search_strategy == 'dfs':
        node = frontier.pop()
      elif self.search_strategy == 'ucs':
        node = heapq.heappop(frontier)

      # 6. Test dell'obiettivo
      vacuum_pos, current_dirts = node.state
      if len(current_dirts) == 0:
        # Obiettivo raggiunto! Ricostruisco la sequenza di azioni (piano)
        return self._reconstruct_path(node)

      # 7. Aggiungo lo stato corrente all'insieme degli esplorati
      explored.add(node.state)

      # 8. Espando il nodo e genero i successori
      #    Azioni: 0=SU, 1=GIÙ, 2=SINISTRA, 3=DESTRA, 4=IDLE, 5=SUCK
      for action in self._get_legal_actions(vacuum_pos, current_dirts):
        next_state, step_cost = self._transition(vacuum_pos, current_dirts, action)

        # Salto i successori già esplorati (evito cicli)
        if next_state in explored:
          continue

        # 9. Creo il nodo figlio
        child_node = SearchNode(
          state=next_state,
          parent=node,
          action=action,
          path_cost=node.path_cost + step_cost,
        )

        # 10. Inserisco il figlio nella frontiera in base alla strategia
        if self.search_strategy == 'bfs':
          frontier.append(child_node)
        elif self.search_strategy == 'dfs':
          frontier.append(child_node)
        elif self.search_strategy == 'ucs':
          heapq.heappush(frontier, child_node)

    return []

  def _get_legal_actions(self, vacuum_pos, current_dirts):
    """Helper che replica env.get_possible_actions() usando lo stato locale."""
    x, y = vacuum_pos
    possible = {
        0: (x, y - 1),
        1: (x, y + 1),
        2: (x - 1, y),
        3: (x + 1, y),
        4: (x, y),
        5: (x, y),
    }
    legal = []
    for action, (nx, ny) in possible.items():
      if action in (4, 5):
        legal.append(action)
        continue
      # Controllo i limiti della griglia e i muri (accessibili tramite env)
      if (0 <= nx < self.env.setting.grid_width and
              0 <= ny < self.env.setting.grid_height and
              (nx, ny) not in self.env.walls):
        legal.append(action)
    return legal

  def _transition(self, vacuum_pos, current_dirts, action):
    """Simula il modello di transizione dell'ambiente T(s, a) -> s'."""
    x, y = vacuum_pos
    directions = {0: (0, -1), 1: (0, 1), 2: (-1, 0), 3: (1, 0)}
    
    new_pos = vacuum_pos
    new_dirts = set(current_dirts)
    step_cost = 1.0  # costo di default

    if action in directions:
      dx, dy = directions[action]
      new_pos = (x + dx, y + dy)
      step_cost = 0.1
    elif action == 5:  
      if vacuum_pos in new_dirts:
        new_dirts.remove(vacuum_pos)
        step_cost = -10.0  # Modifica della ricompensa/costo

    return (new_pos, frozenset(new_dirts)), step_cost

  def _reconstruct_path(self, node):
    """Risalgo i puntatori parent dal goal fino alla radice per formare la sequenza di azioni."""
    path = []
    while node.parent is not None:
      path.append(node.action)
      node = node.parent
    path.reverse()
    return path

Vediamolo in azione!\
Nota: probabilmente non riuscirai nemmeno a vederlo alla difficoltà 0 perché è troppo veloce. Provalo direttamente alla difficoltà 1.

In [34]:
# 1. Creo l'ambiente in difficoltà 1
env = VacuumWorld(difficulty=1)

# 2. Istanzio l'agente di pianificazione con strategia UCS
agent = PlanningAgent(env, "ucs")

# 3. Eseguo un episodio completo
run_episode(env, agent)

197.10000000000008

E per la difficoltà 2?\
La difficoltà 2 inserisce alcune tile speciali dove muoversi costa di più. Il tuo agente trova una soluzione? Pensi che sia una soluzione ottima?

In [39]:
# 1. Creo l'ambiente in difficoltà 2
env = VacuumWorld(difficulty=2)

# 2. Istanzio l'agente di pianificazione con strategia UCS
agent = PlanningAgent(env, "ucs")

# 3. Eseguo un episodio completo
run_episode(env, agent)

ValueError: Unsupported difficulty 2. Supported levels: 0 and 1.